In [1]:
import os
import pandas as pd

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
## PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTE

## svm
from sklearn.svm import SVC
import pandas as pd


In [26]:
ML_CLASSIFIER = ['MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", "XGBoost"] # "ELM"

def classify(X_train, y_train, X_test, y_test, classifier, search_type='grid'):
    """
    Classify the data using a Random Forest Classifier
    :param X_train: Training data
    :param y_train: Training labels
    :param X_test: Testing data
    :param y_test: Testing labels
    :param search_type: Type of search to perform
    :return: Accuracy of the classifier
    """
    # Create a Random Forest Classifier
    #clf = RandomForestClassifier()

    if classifier == "MLP":
        # clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000) ### uncomment if the performance is not good with below hyperparameter
        clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000)
    elif classifier == "GaussianNB": ### {'priors': None, 'var_smoothing': 1e-05}
        clf = GaussianNB(priors=None, var_smoothing= 1e-05) ### Often, the default parameters work well for many datasets.
    elif classifier == "Adaboost":
        dtc = DecisionTreeClassifier(ccp_alpha=0.0, class_weight='balanced', criterion='entropy', max_depth=10, max_features=None, max_leaf_nodes=None, min_impurity_decrease=0.0, min_samples_leaf=2, min_samples_split=2, splitter='best')
        clf = AdaBoostClassifier(estimator=dtc, n_estimators=200, random_state=0, learning_rate=1) ###{'learning_rate': 1, 'n_estimators': 200}
        
    elif classifier == "KNN":
        clf = KNeighborsClassifier(algorithm='auto', leaf_size=10, metric='euclidean', n_jobs=-1,  n_neighbors=1, p=1, weights='uniform')  
    elif classifier == "RFClassifier": ### 
        clf = RandomForestClassifier(bootstrap = False, criterion = 'entropy', max_depth = None, max_features = 'sqrt', min_samples_leaf = 1, min_samples_split = 2, n_estimators = 100, oob_score = False, random_state = 42)
    elif classifier == "SVM_linear": ##
        clf = SVC(kernel='linear')
    elif classifier == "SVM_sigmoid": ###
        clf = SVC(kernel='sigmoid')
    elif classifier == "SVM_RBF": 
        clf = SVC(C=10, cache_size = 200, class_weight = None, gamma = 'scale', kernel = 'rbf', max_iter = -1, probability = True, shrinking = True, tol = 0.001)
    elif classifier == "XGBoost": #'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.7
        clf = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,subsample=0.7)
    else:
        print("Invalid classifier")
        return None
        # Fit the model
    clf.fit(X_train, y_train)

    # Predict the test data
    y_pred = clf.predict(X_test)

    # Calculate the accuracy
   # accuracy = balanced_accuracy_score(y_test, y_pred)
    
    return y_pred


In [27]:
data = pd.read_csv('BT-large-2c-dataset_results_finetune_ALL_Models_v2.csv')
data.head(26)

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,Average
0,resnet50,0.7667,0.8250,0.7850,0.7583,0.7967,0.7267,0.8417,0.5183,0.8800,0.7665
1,resnet101,0.7850,0.8367,0.7050,0.7750,0.8483,0.7183,0.8300,0.5417,0.8733,0.7681
2,densenet121,0.9383,0.9700,0.8500,0.9750,0.9817,0.9417,0.9600,0.9533,0.9733,0.9493
3,densenet169,0.9550,0.9700,0.8183,0.9750,0.9800,0.9383,0.9683,0.9550,0.9783,0.9487
4,vgg16,0.9283,0.9733,0.8300,0.9700,0.9633,0.9333,0.9700,0.9133,0.9817,0.9404
5,vgg19,0.9183,0.9733,0.7367,0.9683,0.9617,0.9383,0.9700,0.8333,0.9800,0.9200
6,alexnet,0.9267,0.9750,0.8250,0.9750,0.9783,0.9583,0.9633,0.6183,0.9833,0.9115
7,resnext50_32x4d,0.7483,0.8167,0.5900,0.7383,0.7717,0.7067,0.8133,0.4833,0.8383,0.7230
8,resnext101_32x8d,0.8083,0.8583,0.6500,0.8267,0.9133,0.7617,0.8633,0.6517,0.9067,0.8044
9,shufflenet_v2_x1_0,0.8717,0.9650,0.7200,0.9317,0.9767,0.8817,0.9650,0.9483,0.9733,0.9148


In [28]:
## sort the data by average
data = data.sort_values(by='Average', ascending=False)
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,Average
14,vit_large_patch16_224,0.9850,0.9967,0.8683,0.9917,0.9850,0.9833,0.9967,0.9833,0.9967,0.9763
24,vit_base_patch32_384,0.9817,0.9917,0.8833,0.9933,0.9850,0.9883,0.9900,0.9650,0.9950,0.9748
22,vit_small_patch32_384,0.9767,0.9917,0.9067,0.9950,0.9900,0.9883,0.9750,0.9483,0.9967,0.9743
13,vit_base_patch32_224,0.9717,0.9917,0.8767,0.9900,0.9883,0.9800,0.9917,0.9633,0.9950,0.9720
20,vit_base_patch16_384,0.9750,0.9850,0.8950,0.9883,0.9867,0.9767,0.9850,0.9633,0.9850,0.9711
15,vit_small_patch32_224,0.9700,0.9850,0.8917,0.9900,0.9933,0.9683,0.9717,0.9367,0.9950,0.9669
12,vit_base_patch16_224,0.9733,0.9933,0.8650,0.9817,0.9867,0.9783,0.9917,0.9383,0.9900,0.9665
23,vit_small_patch16_384,0.9767,0.9917,0.8333,0.9900,0.9900,0.9900,0.9750,0.9450,0.9917,0.9648
19,vit_small_patch16_224,0.9700,0.9917,0.8583,0.9900,0.9850,0.9800,0.9750,0.9383,0.9933,0.9646
17,vit_base_patch8_224,0.9683,0.9933,0.8550,0.9850,0.9850,0.9733,0.9950,0.8817,0.9933,0.9589


In [29]:
## change the indeces of the data to Model
data = data.set_index('Model')
data

,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,Average
Model,,,,,,,,,,
vit_large_patch16_224,0.9850,0.9967,0.8683,0.9917,0.9850,0.9833,0.9967,0.9833,0.9967,0.9763
vit_base_patch32_384,0.9817,0.9917,0.8833,0.9933,0.9850,0.9883,0.9900,0.9650,0.9950,0.9748
vit_small_patch32_384,0.9767,0.9917,0.9067,0.9950,0.9900,0.9883,0.9750,0.9483,0.9967,0.9743
vit_base_patch32_224,0.9717,0.9917,0.8767,0.9900,0.9883,0.9800,0.9917,0.9633,0.9950,0.9720
vit_base_patch16_384,0.9750,0.9850,0.8950,0.9883,0.9867,0.9767,0.9850,0.9633,0.9850,0.9711
vit_small_patch32_224,0.9700,0.9850,0.8917,0.9900,0.9933,0.9683,0.9717,0.9367,0.9950,0.9669
vit_base_patch16_224,0.9733,0.9933,0.8650,0.9817,0.9867,0.9783,0.9917,0.9383,0.9900,0.9665
vit_small_patch16_384,0.9767,0.9917,0.8333,0.9900,0.9900,0.9900,0.9750,0.9450,0.9917,0.9648
vit_small_patch16_224,0.9700,0.9917,0.8583,0.9900,0.9850,0.9800,0.9750,0.9383,0.9933,0.9646


In [30]:
data = data.sort_values(by = 'Average', axis = 1) 
data

,GaussianNB,SVM_sigmoid,XGBoost,RFClassifier,Adaboost,SVM_linear,KNN,MLP,SVM_RBF,Average
Model,,,,,,,,,,
vit_large_patch16_224,0.8683,0.9833,0.9850,0.9833,0.9917,0.9967,0.9850,0.9967,0.9967,0.9763
vit_base_patch32_384,0.8833,0.9650,0.9817,0.9883,0.9933,0.9900,0.9850,0.9917,0.9950,0.9748
vit_small_patch32_384,0.9067,0.9483,0.9767,0.9883,0.9950,0.9750,0.9900,0.9917,0.9967,0.9743
vit_base_patch32_224,0.8767,0.9633,0.9717,0.9800,0.9900,0.9917,0.9883,0.9917,0.9950,0.9720
vit_base_patch16_384,0.8950,0.9633,0.9750,0.9767,0.9883,0.9850,0.9867,0.9850,0.9850,0.9711
vit_small_patch32_224,0.8917,0.9367,0.9700,0.9683,0.9900,0.9717,0.9933,0.9850,0.9950,0.9669
vit_base_patch16_224,0.8650,0.9383,0.9733,0.9783,0.9817,0.9917,0.9867,0.9933,0.9900,0.9665
vit_small_patch16_384,0.8333,0.9450,0.9767,0.9900,0.9900,0.9750,0.9900,0.9917,0.9917,0.9648
vit_small_patch16_224,0.8583,0.9383,0.9700,0.9800,0.9900,0.9750,0.9850,0.9917,0.9933,0.9646


In [34]:
list(data.head(5).index)

['vit_large_patch16_224',
 'vit_base_patch32_384',
 'vit_small_patch32_384',
 'vit_base_patch32_224',
 'vit_base_patch16_384']

In [32]:
list(data.columns[-4:-1])

['KNN', 'MLP', 'SVM_RBF']

### Top 3 performance model

In [36]:
top_ml_model_combinations = [
                          ['KNN', 'MLP'],
                          ['KNN', 'SVM_RBF'],
                          ['MLP', 'SVM_RBF'],
                          ['KNN', "SVM_RBF", "MLP"]
                          ]

Ensemble vgg16 with Top 2 networks ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']

In [33]:



# top_model_combinations = [
#                           ['vit_base_patch32_384', 'vit_small_patch32_224'],
#                           ['vit_base_patch32_384', 'vgg16'],
#                           ['vit_small_patch32_224', 'vgg16'],
#                           ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']
#                           ]

In [47]:
columns = ['Model', 'vit_large_patch16_224',
                    'vit_base_patch32_384',
                    'vit_small_patch32_384',
                    'vit_base_patch32_224',
                    'vit_base_patch16_384']

dataframe = pd.DataFrame(columns=columns) ### new empty pandas DataFrame with specified column names
# add 12 rows to the dataframe with zero values 
for model_list in top_ml_model_combinations:
    model_name = ' + '.join(model_list)
    new_row = {}
    new_row['Model'] = model_name
    for col in columns[1:]:
        new_row[col] = 0
    dataframe.loc[len(dataframe)] = new_row

model_versions = ['simple_version', "with_normalization_&_PCA", "with_SMOTE_only", "with_normalization_&_PCA_SMOTE"]

model_version_index = 3
main_path = 'extracted_features_BT-large-2c'
for feature_extractor in columns[1:]:
    
    print('Deep Feature Extractor List:', feature_extractor)
    
    sub_dir = os.path.join(main_path, feature_extractor)
    X_train = np.load(os.path.join(sub_dir, 'train_data_array_features.npy'))
    y_train = np.load(os.path.join(sub_dir, 'train_data_array_labels.npy'))
    X_test = np.load(os.path.join(sub_dir, 'test_data_array_features.npy'))
    y_test = np.load(os.path.join(sub_dir, 'test_data_array_labels.npy'))

    X_train = np.squeeze(X_train, axis=1)
    X_test = np.squeeze(X_test, axis=1)

    # # apply the standard scaler and PCA
    scaler = StandardScaler()
    X_train_normalized = scaler.fit_transform(X_train)
    X_test_normalized = scaler.transform(X_test)

    # Step 2 & 3: Apply PCA
    n_components = int(0.45 * X_train.shape[1])
    pca = PCA(n_components=n_components)
    X_train = pca.fit_transform(X_train_normalized)
    X_test = pca.transform(X_test_normalized)

    ## no of example in each class
    print("no of example in each class before SMOTE:", np.bincount(y_train))


    # # ## over sample the data (data augmentation)
    smote = SMOTE(sampling_strategy='minority')
    X_train, y_train = smote.fit_resample(X_train, y_train)

    # ## no of example in each class
    print("no of example in each class after SMOTE:", np.bincount(y_train))
    
    for model_list in top_ml_model_combinations:
        all_predicted_probs = []
        for ml_classifier in model_list:
            probs = classify(X_train, y_train, X_test, y_test, ml_classifier)
            all_predicted_probs.append(probs)
        
        ## take the average of the probabilities
        all_predicted_probs = np.array(all_predicted_probs)
        all_predicted_probs = np.mean(all_predicted_probs, axis=0)
        ## if prob is greater than 0.5 then 1 else 0
        y_pred = np.where(all_predicted_probs >= 0.5, 1, 0)
        accuracy = balanced_accuracy_score(y_test, y_pred)
        accuracy = float(round(accuracy, 4))
        print('Accuracy:', accuracy)
    
        model_name = ' + '.join(model_list)

        dataframe.loc[dataframe['Model'] == model_name, feature_extractor] = accuracy

    print(dataframe)
    dataframe.to_csv(f'BT-large-2c-dataset_results_top_three_{model_versions[model_version_index]}_ml_ensemble.csv', index=False)


KeyboardInterrupt: 

In [43]:
X_train.shape

(8536, 460)

In [42]:
all_predicted_probs.shape

(600,)

In [19]:
n_components

768

In [14]:
dataframe

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF
0,vit_base_patch32_384 + vit_small_patch32_224,0.736086,0.762759,0.334131,0.731610,0.771137,0.738784,0.799189,0.683773,0.754797
1,vit_base_patch32_384 + vit_small_patch16_224,0.733586,0.758041,0.333893,0.754932,0.796542,0.748919,0.779797,0.658766,0.754797
2,vit_small_patch32_224 + vit_small_patch16_224,0.740533,0.756554,0.410204,0.764571,0.812421,0.777488,0.719953,0.619690,0.765811
3,vit_base_patch32_384 + vit_small_patch32_224 +...,0.721848,0.767432,0.325231,0.747432,0.802421,0.782297,0.771554,0.665940,0.762297
